In [1]:
# --- Step 1: Load the Model and Necessary Imports ---
import os
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score
from tensorflow.keras.models import load_model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# Load the model
model_path = "model_finetuned_3_15.h5"  # Replace with your model file path
model = load_model(model_path, compile=False)  # Load without compiling to modify loss later if needed
print(f"Model loaded successfully from {model_path}.")

fpi_dist_folder=r"C:\Users\ryane\Desktop\Aurora_Project\MMS-FPI-Data-Gaps\data_03_15\fpi"
fgm_folder=r"C:\Users\ryane\Desktop\Aurora_Project\MMS-FPI-Data-Gaps\data_03_15\fgm"
fpi_moms_folder=r"C:\Users\ryane\Desktop\Aurora_Project\MMS-FPI-Data-Gaps\data_03_15\moms"

print("Folders set up for evaluation:")
print(f"FPI Dist Folder: {fpi_dist_folder}")
print(f"FGM Folder: {fgm_folder}")
print(f"FPI Moms Folder: {fpi_moms_folder}")





Model loaded successfully from model_finetuned_3_15.h5.
Folders set up for evaluation:
FPI Dist Folder: C:\Users\ryane\Desktop\Aurora_Project\MMS-FPI-Data-Gaps\data_03_15\fpi
FGM Folder: C:\Users\ryane\Desktop\Aurora_Project\MMS-FPI-Data-Gaps\data_03_15\fgm
FPI Moms Folder: C:\Users\ryane\Desktop\Aurora_Project\MMS-FPI-Data-Gaps\data_03_15\moms


In [2]:
def match_files(fpi_dist_folder, fgm_folder, fpi_moms_folder):
    # This function attempts to match files by timestamp.
    fpi_dist_files = [f for f in os.listdir(fpi_dist_folder) if f.endswith('.cdf')]
    matched_files = []
    for fpi_dist_filename in fpi_dist_files:
        fpi_dist_timestamp = extract_timestamp(fpi_dist_filename)
        if fpi_dist_timestamp is None:
            continue

        fgm_files = [f for f in os.listdir(fgm_folder) if f.endswith('.cdf')]
        fgm_match = [f for f in fgm_files if extract_timestamp(f) == fpi_dist_timestamp]
        if not fgm_match:
            continue
        fgm_filename = fgm_match[0]

        fpi_moms_files = [f for f in os.listdir(fpi_moms_folder) if f.endswith('.cdf')]
        moms_match = [f for f in fpi_moms_files if extract_timestamp(f) == fpi_dist_timestamp]
        if not moms_match:
            continue
        fpi_moms_filename = moms_match[0]

        matched_files.append((fpi_dist_timestamp, fpi_dist_filename, fgm_filename, fpi_moms_filename))

    return matched_files

In [5]:
import numpy as np
import os
import datetime
import cdflib
from scipy.interpolate import interp1d

# --- Define Necessary Helper Functions ---

def extract_timestamp(filename):
    """Extract timestamp from filename (YYYYMMDDHHMMSS)."""
    try:
        parts = filename.split('_')
        for part in parts:
            if part.isdigit() and len(part) == 14:
                return datetime.datetime.strptime(part, "%Y%m%d%H%M%S")
        raise ValueError("No valid timestamp found in filename: " + filename)
    except Exception as e:
        print(f"Error extracting timestamp from filename: {filename} - {e}")
        return None

def load_fgm_data(filepath):
    """Load magnetic field data from FGM CDF file."""
    with cdflib.CDF(filepath) as fgm_cdf:
        epoch = fgm_cdf.varget('Epoch')
        b_gse = fgm_cdf.varget('mms1_fgm_b_gse_brst_l2')  # shape: (time, 4)
    return epoch, b_gse

def load_fpi_dist_data(filepath):
    """Load distribution data from FPI CDF file."""
    with cdflib.CDF(filepath) as fpi_cdf:
        epoch = fpi_cdf.varget('Epoch')
        phi = fpi_cdf.varget('mms1_des_phi_brst')     # (time, 32)
        theta = fpi_cdf.varget('mms1_des_theta_brst') # (16,)
        energy = fpi_cdf.varget('mms1_des_energy_brst') # (time, 32)
        psd = fpi_cdf.varget('mms1_des_dist_brst')    # (time, 32,16,32)
    return epoch, theta, phi, energy, psd

def interpolate_b_field(fgm_epoch, b_gse, fpi_epoch):
    """Interpolate magnetic field data to FPI timestamps."""
    b_gse_comp = b_gse[:, :3]
    interpolator = interp1d(fgm_epoch, b_gse_comp, axis=0, bounds_error=False, fill_value="extrapolate")
    return interpolator(fpi_epoch)

def calculate_single_pitch_angle(phi_bins, theta_bins, b_vec):
    """Calculate pitch angle for a single time step."""
    phi_rad = np.deg2rad(phi_bins)
    theta_rad = np.deg2rad(theta_bins)
    phi_grid, theta_grid = np.meshgrid(phi_rad, theta_rad, indexing='ij') # (32,16)
    look_x = np.sin(theta_grid) * np.cos(phi_grid)
    look_y = np.sin(theta_grid) * np.sin(phi_grid)
    look_z = np.cos(theta_grid)
    look_direction = np.stack([look_x, look_y, look_z], axis=2) # (32,16,3)
    look_flat = look_direction.reshape(-1, 3) # (32*16,3)
    mag = np.linalg.norm(b_vec)
    b_unit = b_vec / mag if mag != 0 else np.zeros(3)
    v_dot_b = np.dot(look_flat, b_unit)
    v_dot_b = np.clip(v_dot_b, -1.0, 1.0)
    angles = np.arccos(v_dot_b) * (180.0 / np.pi)
    return angles.reshape(32,16)

def prepare_data(fgm_filepath, fpi_dist_filepath):
    """Prepare input and target data."""
    fgm_epoch, b_gse = load_fgm_data(fgm_filepath)
    fpi_epoch, theta, phi, energy, psd = load_fpi_dist_data(fpi_dist_filepath)
    b_gse_interp = interpolate_b_field(fgm_epoch, b_gse, fpi_epoch)
    time_steps = psd.shape[0]
    y_data = psd
    X_data_list = []
    for t in range(time_steps):
        phi_bins = phi[t]
        energy_bins = energy[t]
        pitch_angles_t = calculate_single_pitch_angle(phi_bins, theta, b_gse_interp[t]) # (32,16)
        phi_grid = np.tile(phi_bins[:, np.newaxis], (1, theta.size)).repeat(energy_bins.size, axis=2)
        theta_grid = np.tile(theta[np.newaxis, :], (phi_bins.size, 1)).repeat(energy_bins.size, axis=2)
        energy_grid = np.tile(energy_bins[np.newaxis, np.newaxis, :], (phi_bins.size, theta.size, 1))
        pitch_angle_grid = np.repeat(pitch_angles_t[..., np.newaxis], energy_bins.size, axis=2)
        X_t = np.stack([phi_grid, theta_grid, energy_grid, pitch_angle_grid], axis=-1)
        X_data_list.append(X_t)
    X_data = np.array(X_data_list, dtype=np.float32) # (time,32,16,32,4)
    return X_data, y_data

def split_data(X_data, y_data, train_ratio=0.8, val_ratio=0.1):
    """Split data into train, validation, and test sets."""
    total_samples = X_data.shape[0]
    train_size = int(train_ratio * total_samples)
    val_size = int(val_ratio * total_samples)
    X_train = X_data[:train_size]
    y_train = y_data[:train_size]
    X_val = X_data[train_size:train_size+val_size]
    y_val = y_data[train_size:train_size+val_size]
    X_test = X_data[train_size+val_size:]
    y_test = y_data[train_size+val_size:]
    return X_train, y_train, X_val, y_val, X_test, y_test

def match_files(fpi_dist_folder, fgm_folder, fpi_moms_folder):
    """Match files by timestamp."""
    fpi_dist_files = [f for f in os.listdir(fpi_dist_folder) if f.endswith('.cdf')]
    matched_files = []
    for fpi_dist_filename in fpi_dist_files:
        fpi_dist_timestamp = extract_timestamp(fpi_dist_filename)
        if fpi_dist_timestamp is None:
            continue
        fgm_files = [f for f in os.listdir(fgm_folder) if f.endswith('.cdf')]
        fgm_match = [f for f in fgm_files if extract_timestamp(f) == fpi_dist_timestamp]
        if not fgm_match:
            continue
        fgm_filename = fgm_match[0]
        fpi_moms_files = [f for f in os.listdir(fpi_moms_folder) if f.endswith('.cdf')]
        moms_match = [f for f in fpi_moms_files if extract_timestamp(f) == fpi_dist_timestamp]
        if not moms_match:
            continue
        fpi_moms_filename = moms_match[0]
        matched_files.append((fpi_dist_timestamp, fpi_dist_filename, fgm_filename, fpi_moms_filename))
    return matched_files

# --- Step 2: Prepare Matched Files and Data ---

fpi_dist_folder = "C://Users//ryane//Desktop//Aurora_Project//MMS-FPI-Data-Gaps//data//mms1//fpi//brst//l2//des-dist//2024//03//01"
fgm_folder = "C://Users//ryane//Desktop//Aurora_Project//MMS-FPI-Data-Gaps//data//mms1//fgm"
fpi_moms_folder = "C://Users//ryane//Desktop//Aurora_Project//MMS-FPI-Data-Gaps//data//mms1//fpi//brst//l2//des-moms//2024//03//01"

print("Matching files...")
matched_files = match_files(fpi_dist_folder, fgm_folder, fpi_moms_folder)
print(f"Found {len(matched_files)} matched file sets.")

if not matched_files:
    print("No matched files found.")
    exit()

print("\nPreparing data for evaluation...")
fpi_dist_timestamp, fpi_dist_filename, fgm_filename, fpi_moms_filename = matched_files[0]
fpi_dist_filepath = os.path.join(fpi_dist_folder, fpi_dist_filename)
fgm_filepath = os.path.join(fgm_folder, fgm_filename)

X_data, y_data = prepare_data(fgm_filepath, fpi_dist_filepath)
X_train, y_train, X_val, y_val, X_test, y_test = split_data(X_data, y_data)

# Expand time dimension
X_val = np.expand_dims(X_val, axis=1)
y_val = np.expand_dims(y_val, axis=1)
X_test = np.expand_dims(X_test, axis=1)
y_test = np.expand_dims(y_test, axis=1)


Matching files...
Found 5 matched file sets.

Preparing data for evaluation...


AxisError: axis 2 is out of bounds for array of dimension 2

In [4]:

# Match files
matched_files = match_files(fpi_dist_folder, fgm_folder, fpi_moms_folder)
print(f"Found {len(matched_files)} matched file sets.")


NameError: name 'extract_timestamp' is not defined